# Seed 22: failure-start versus random at round 2
Test whether the seed-11 round-two advantage repeats with a different training seed. Train **only a new baseline, random acquisition, and failure-start acquisition**. No Demo-SCORE, original targeted, diversity, or continued-training arms.

Primary comparison fixed before this run: **failure-start minus random task success at round 2**, with 90 expert episodes and 40k cumulative updates per arm. One new baseline is shared by the arms. Existing seed-11 checkpoints are not used as training parents. The new seed controls initialization, sampling, and the initial demonstration subset; this tests run-level variability, not optimizer randomness alone.

Default cost: 60k total new optimizer updates (20k baseline + 20k per arm), 40 diagnostic rollouts for selection, and 1,000 evaluation rollouts across five checkpoints. Original DDPM / execute=8. Evaluation starts match seed 11, so this is a seed replication on a reused evaluation suite, not an untouched confirmation. Keep the selector fixed, report both rounds, and do not choose a winner retrospectively.


In [ ]:
from pathlib import Path
import subprocess, sys
REPOSITORY = "https://github.com/0-5-blood-prince/robotics-data-flywheel.git"
REVISION = "main"  # Set to a commit SHA to reproduce a particular code version.
ROOT = Path('/content/robotics-data-flywheel')
if any(name.startswith('robot_flywheel') for name in sys.modules):
    raise RuntimeError('Restart the runtime before refreshing project code. Saved Drive checkpoints remain available.')
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(ROOT)], check=True)
else:
    if not (ROOT / '.git').exists():
        raise RuntimeError('This directory came from an old ZIP. Start a fresh Colab runtime once to switch to GitHub setup.')
    changes = subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'],text=True)
    if changes.strip(): raise RuntimeError('Local code changes found; preserve them before refreshing.')
subprocess.run(['git','-C',str(ROOT),'fetch','origin',REVISION],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
CODE_COMMIT = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('Code commit:', CODE_COMMIT)
import os, importlib
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{ROOT}[simulation]'], check=True)
# Activate src in this already-running kernel: editable-install .pth files
# are normally processed only when a Python interpreter starts.
sys.path.insert(0, str(ROOT / 'src'))
importlib.invalidate_caches()
import robot_flywheel
assert Path(robot_flywheel.__file__).resolve().is_relative_to(ROOT.resolve())
print('Package ready:', robot_flywheel.__file__)

# Check the native simulator API before dataset preparation.
subprocess.run([sys.executable, '-c',
    "import gymnasium as gym, gym_pusht; e=gym.make('gym_pusht/PushT-v0'); e.reset(seed=0); e.step(e.action_space.sample()); e.close(); print('Simulator ready')"], check=True)


## Persistence and runtime
Set `USE_DRIVE=True` to preserve checkpoints across runtime replacement. Checkpoints are saved every 1,000 updates; rerunning training resumes automatically. Only load checkpoints produced by this project or another trusted source.

In [ ]:
import os, sys, json, torch, numpy as np
os.environ['SDL_VIDEODRIVER'] = 'dummy'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/robotics-data-flywheel-runs')
else:
    WORK = ROOT / 'runs'
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'code_commit.txt').write_text(CODE_COMMIT)
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.cuda.is_available(),
       'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})
if not torch.cuda.is_available():
    print('Switch to a GPU runtime before substantial training. CPU is suitable for tests only.')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Load data and verify simulator compatibility

In [ ]:
from robot_flywheel.prepare import prepare
from robot_flywheel.data import load_zarr
EPISODES, SOURCE = prepare(ROOT / 'data/pusht')
display(SOURCE)


In [ ]:
import matplotlib.pyplot as plt
from robot_flywheel.rollout import check_replay
COMPATIBILITY = check_replay(EPISODES, WORK / 'compatibility', count=3)
display(COMPATIBILITY)
fig, axs = plt.subplots(1,3,figsize=(15,4))
for ax, episode in zip(axs, EPISODES[:3]):
    d = np.load(WORK / 'compatibility' / f'replay_{episode.id}.npz')
    ax.plot(d['recorded'][:,2],d['recorded'][:,3],label='Recorded object')
    ax.plot(d['simulated'][:,2],d['simulated'][:,3],label='Replayed object')
    ax.set(title=f'Episode {episode.id}',aspect='equal'); ax.legend()
plt.show()
assert COMPATIBILITY['passed'], 'Compatibility failed. Inspect report/traces before training.'


## 1. Freeze the new seed and primary comparison
Inherit model and budget settings from the original seed-11 protocol, change training seed to 22 and rounds to 2. Both methods share the new baseline and paired evaluation starts. Output directories are separate from all earlier runs. The failure-start selector remains fixed at 50% failure matching and 50% coverage fill (unused matching slots go to coverage).

The original learning-gate metadata is inherited for comparability; it is not new evidence of seed-22 baseline competence. Inspect baseline results as part of this replication. Do not switch back to a favorable seed if results regress.


In [ ]:
from dataclasses import replace,asdict
from robot_flywheel.experiment import Protocol
from robot_flywheel.replication import run_random_reference
from robot_flywheel.initial_state import StartConfig,run_failure_start,compare_failure_start
from robot_flywheel.artifacts import write_json
SEED11 = WORK/'exploratory_ddpm_v1_seed11'
NEW_SEED = 22
saved_protocol=json.loads((SEED11/'frozen_protocol.json').read_text())['protocol']
assert NEW_SEED not in saved_protocol['training_seeds'], 'Choose a new training seed'
PROTOCOL=replace(Protocol(**saved_protocol),training_seeds=[NEW_SEED],rounds=2)
CONFIG=StartConfig(failure_fraction=.5)
REFERENCE=WORK/f'failure_random_replication_seed{NEW_SEED}'/'reference'
OUTPUT=WORK/f'failure_random_replication_seed{NEW_SEED}'/'failure_start'
PLAN=OUTPUT.parent/'comparison_plan.json'
plan=dict(training_seed=NEW_SEED,primary_round=2,primary_metric='failure_start minus random task success',
          protocol=asdict(PROTOCOL),selector=asdict(CONFIG),evaluation='Reused paired starts; seed replication, exploratory')
if PLAN.exists() and json.loads(PLAN.read_text())!=plan:
    raise ValueError('Comparison plan changed; use new output directories')
write_json(PLAN,plan)
display(plan)


## 2. Train the new shared baseline and random control
This builds the seed-22 baseline from scratch, then two sequential random acquisitions. Each stage saves optimizer, EMA, normalization, and RNG state. Rerunning resumes interrupted work. No existing seed-11 experiment is touched.

In [ ]:
RANDOM_CHECKPOINTS=run_random_reference(
    EPISODES,SOURCE,PROTOCOL,REFERENCE,COMPATIBILITY,final_evaluation=False)
display(RANDOM_CHECKPOINTS)


## 3. Train failure-start from that same new baseline
Each round gathers 20 diagnostic rollouts from its current parent, then adds 20 expert episodes. With at least ten failures, ten acquisitions match failed starts and ten maximize initial-state coverage. The random arm needs no diagnostic rollouts because it does not use outcomes; this extra selection cost is explicit.

In [ ]:
FAILURE_CHECKPOINTS=run_failure_start(
    EPISODES,SOURCE,REFERENCE,OUTPUT,COMPATIBILITY,CONFIG,
    rounds_to_run=PROTOCOL.rounds,final_evaluation=False)
display(FAILURE_CHECKPOINTS)


## 4. Verify shared baseline and matched budgets before evaluation

In [ ]:
from robot_flywheel.initial_state import sha
base=REFERENCE/f'seed_{NEW_SEED}'/'baseline/latest.pt'
base_hash=sha(base)
for arm,home in [('random',REFERENCE),('failure_start',OUTPUT)]:
    first=json.loads((home/f'seed_{NEW_SEED}'/arm/'round_1/selection.json').read_text())
    assert first['parent_checkpoint']==base_hash
    for r in range(1,PROTOCOL.rounds+1):
        checkpoint=torch.load(home/f'seed_{NEW_SEED}'/arm/f'round_{r}/latest.pt',map_location='cpu',weights_only=False)
        assert checkpoint['update']==PROTOCOL.baseline_updates+r*PROTOCOL.round_updates
        assert len(checkpoint['manifest']['episode_ids'])==PROTOCOL.initial_episodes+r*PROTOCOL.acquire
        assert checkpoint['config']['seed']==NEW_SEED
print('Shared seed-22 baseline and matched episode/update budgets verified.')


## 5. Evaluate the five new checkpoints
All evaluations use identical starts and deterministic inference seeds. Cached episodes resume after interruption. The primary endpoint remains round two regardless of the round-one result.

In [ ]:
RANDOM_RESULTS=run_random_reference(
    EPISODES,SOURCE,PROTOCOL,REFERENCE,COMPATIBILITY,final_evaluation=True)
FAILURE_RESULTS=run_failure_start(
    EPISODES,SOURCE,REFERENCE,OUTPUT,COMPATIBILITY,CONFIG,
    rounds_to_run=PROTOCOL.rounds,final_evaluation=True)


## 6. Primary result: round two
Paired bootstrap intervals here reflect evaluation-start uncertainty **within seed 22**, not uncertainty across many trained models. Compare the sign and size with seed 11, but two seeds still provide limited evidence of robustness. Report failure-start and random separately; do not attribute baseline-to-final gains solely to the selector.


In [ ]:
import pandas as pd
COMPARISONS,fig=compare_failure_start(REFERENCE,OUTPUT)
plt.show();display(COMPARISONS)
results=pd.read_csv(OUTPUT/'combined_results.csv')
display(results[['seed','arm','round','episodes','transitions','success_rate','mean_max_coverage']])
assert json.loads((OUTPUT/'comparisons.json').read_text())['round']==2


## 7. Side-by-side seed summary
Read the actual seed-11 saved results if available; never hardcode its measured rates. This table is descriptive, not a pooled significance claim. Both runs use the same evaluation suite.

In [ ]:
rows=[]
old_failure=WORK/'failure_start_v1_seed11'/'results.json'
if old_failure.exists():
    old_random=json.loads((SEED11/'results.json').read_text())['records']
    old_new=json.loads(old_failure.read_text())['records']
    random_r=next(x for x in old_random if x['arm']=='random' and x['round']==2)
    failure_r=next(x for x in old_new if x['arm']=='failure_start' and x['round']==2)
    rows.append(dict(seed=random_r['seed'],round=2,random=random_r['success_rate'],
        failure_start=failure_r['success_rate'],difference_pp=100*(failure_r['success_rate']-random_r['success_rate'])))
random_r=next(x for x in RANDOM_RESULTS if x['arm']=='random' and x['round']==2)
failure_r=next(x for x in FAILURE_RESULTS if x['round']==2)
rows.append(dict(seed=NEW_SEED,round=2,random=random_r['success_rate'],
    failure_start=failure_r['success_rate'],difference_pp=100*(failure_r['success_rate']-random_r['success_rate'])))
seed_summary=pd.DataFrame(rows)
display(seed_summary)
seed_summary.to_csv(OUTPUT/'seed_summary.csv',index=False)
from IPython.display import FileLink
for name in ['combined_results.csv','comparisons.json','combined_curves.png','seed_summary.csv']:
    display(FileLink(str(OUTPUT/name)))
